In [2]:
"""
Day 13: SIEM Log Analysis for SE Attack Detection
Parses multi-source security log streams to identify credential attacks,
anomalous logins, and post-exploitation persistence rules.
"""

from collections import Counter
import datetime
import json
import re

# Expanded SIEM Log Sample containing realistic SE attack telemetry
LOG_SAMPLE = """
2026-09-17 02:34:10 FAILED_LOGIN user=admin ip=45.33.32.156
2026-09-17 02:34:12 FAILED_LOGIN user=admin ip=45.33.32.156
2026-09-17 02:34:14 FAILED_LOGIN user=admin ip=45.33.32.156
2026-09-17 02:34:16 SUCCESS_LOGIN user=admin ip=45.33.32.156
2026-09-17 02:35:00 EMAIL_RULE_CREATED user=admin rule=forward_all destination=attacker@exfiltrate-data.com
2026-09-17 08:00:01 SUCCESS_LOGIN user=riya ip=192.168.1.10
2026-09-17 08:15:22 FAILED_LOGIN user=john ip=192.168.1.45
2026-09-17 09:05:11 EMAIL_RULE_CREATED user=john rule=move_to_trash keyword=invoice
2026-09-17 03:12:00 SUCCESS_LOGIN user=ceo ip=185.220.101.5
"""


def parse_and_analyze(logs):
    print("=" * 65)
    print("SIEM LOG ANALYSIS ENGINE — SOCIAL ENGINEERING & ANOMALY DETECTION")
    print("=" * 65)

    alerts = []

    # 1. Detect Brute Force / Credential Stuffing Attempts (>= 3 failures per user)
    failed_logins = re.findall(
        r"(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}) FAILED_LOGIN user=(\w+) ip=([\d.]+)",
        logs,
    )
    fail_counts = Counter(user for _, user, _ in failed_logins)

    for user, count in fail_counts.items():
        if count >= 3:
            # Extract IPs involved
            ips = list(
                {ip for _, u, ip in failed_logins if u == user}
            )
            alert = {
                "type": "BRUTE_FORCE_DETECTED",
                "severity": "HIGH",
                "user": user,
                "details": f"{count} failed login attempts detected from IP(s): {', '.join(ips)}",
            }
            alerts.append(alert)
            print(
                f"🚨 [HIGH ALERT] Brute Force / Password Spraying: User '{user}' ({count} failures from {', '.join(ips)})"
            )

    # 2. Detect Suspicious Inbox Rules (Post-Phishing Persistence Mechanism)
    rule_events = re.findall(
        r"(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}) EMAIL_RULE_CREATED user=(\w+) rule=(\w+)(?: destination=([\w@.-]+))?",
        logs,
    )
    for timestamp, user, rule, dest in rule_events:
        details = f"Rule '{rule}' created at {timestamp}"
        if dest:
            details += f" forwarding to external destination: {dest}"

        alert = {
            "type": "SUSPICIOUS_EMAIL_RULE",
            "severity": "CRITICAL" if "forward" in rule else "MEDIUM",
            "user": user,
            "details": details,
        }
        alerts.append(alert)
        severity_icon = "🚨 [CRITICAL]" if "forward" in rule else "⚠️ [MEDIUM]"
        print(
            f"{severity_icon} Post-Compromise Persistence: User '{user}' created email rule '{rule}'"
        )

    # 3. Detect Anomalous / Off-Hours Logins (e.g., logins between 00:00 and 05:59 AM)
    success_logins = re.findall(
        r"(\d{4}-\d{2}-\d{2} (\d{2}):\d{2}:\d{2}) SUCCESS_LOGIN user=(\w+) ip=([\d.]+)",
        logs,
    )
    for timestamp, hour, user, ip in success_logins:
        if 0 <= int(hour) < 6:
            alert = {
                "type": "OFF_HOURS_SUCCESSFUL_LOGIN",
                "severity": "MEDIUM",
                "user": user,
                "details": f"Successful login recorded at off-hours time ({timestamp}) from IP {ip}",
            }
            alerts.append(alert)
            print(
                f"⚠️ [MEDIUM ALERT] Anomalous Off-Hours Access: User '{user}' logged in at {timestamp} from {ip}"
            )

    print("\n" + "=" * 65)
    print(f"SUMMARY: {len(alerts)} Security Alerts Generated.")
    print("=" * 65)

    # Export alerts to JSON file for audit SIEM logs
    with open("siem_alerts.json", "w") as f:
        json.dump(alerts, f, indent=4)
    print("Alert report exported to 'siem_alerts.json'.")


if __name__ == "__main__":
    parse_and_analyze(LOG_SAMPLE)

SIEM LOG ANALYSIS ENGINE — SOCIAL ENGINEERING & ANOMALY DETECTION
🚨 [HIGH ALERT] Brute Force / Password Spraying: User 'admin' (3 failures from 45.33.32.156)
🚨 [CRITICAL] Post-Compromise Persistence: User 'admin' created email rule 'forward_all'
⚠️ [MEDIUM] Post-Compromise Persistence: User 'john' created email rule 'move_to_trash'
⚠️ [MEDIUM ALERT] Anomalous Off-Hours Access: User 'admin' logged in at 2026-09-17 02:34:16 from 45.33.32.156
⚠️ [MEDIUM ALERT] Anomalous Off-Hours Access: User 'ceo' logged in at 2026-09-17 03:12:00 from 185.220.101.5

SUMMARY: 5 Security Alerts Generated.
Alert report exported to 'siem_alerts.json'.
